# RiskLens: Bureau data

**Goal of this notebook:** `bureau.csv` has the past credits each applicant had at *other* banks (reported by the credit bureau). I want to turn it into a few useful features, one row per applicant, and check if they relate to default.

**The problem:** the main table has one row per applicant, but the bureau table has many rows per applicant. I can't just join them or the applicants would get duplicated, so I summarise it first.

**What I'll do**
1. Look at the raw bureau table
2. Count how many past credits each person has
3. Summarise per applicant (`groupby`)
4. Check the relation with `TARGET`

## 1. Loading the bureau table

In [5]:
import pandas as pd

path = "../data/raw/home-credit-default-risk/"
bureau = pd.read_csv(path + "bureau.csv")
print(bureau.shape)
bureau.head()

(1716428, 17)


,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,91323.0,0.0,NaN,0.0,Consumer credit,-131,NaN
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,225000.0,171342.0,NaN,0.0,Credit card,-20,NaN
2,215354,5714464,Active,currency 1,-203,0,528.0,NaN,NaN,0,464323.5,NaN,NaN,0.0,Consumer credit,-16,NaN
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0,90000.0,NaN,NaN,0.0,Credit card,-16,NaN
4,215354,5714466,Active,currency 1,-629,0,1197.0,NaN,77674.5,0,2700000.0,NaN,NaN,0.0,Consumer credit,-21,NaN


The bureau table has **1,716,428 rows and 17 columns**. Each row is **one past credit** (credit card, consumer credit, etc.), not one person. That's why it's so much bigger than the main table (307,511 rows).

Looking at the first rows, a few columns have a lot of missing values (`AMT_CREDIT_SUM_DEBT`, `AMT_CREDIT_SUM_LIMIT`, `AMT_ANNUITY`). Also, the `DAYS_` columns are negative again, same as in the main table (days before the application).

## 2. How many past credits per person?

In [6]:
# how many past credits does each person have?
print(bureau["SK_ID_CURR"].value_counts().describe())

count    305811.000000
mean          5.612709
std           4.430354
min           1.000000
25%           2.000000
50%           4.000000
75%           8.000000
max         116.000000
Name: count, dtype: float64


| | Past credits per person |
|---|---|
| People with bureau history | 305,811 |
| Average | 5.6 |
| Median | 4 |
| Min / Max | 1 / 116 |

Most people have between 2 and 8 past credits. The average (5.6) is higher than the median (4) because a few people have a lot of credits (up to 116), so the data is skewed.

The 305,811 includes applicants from the test set as well, that's why this number doesn't match the train table directly. I'll see how many train applicants are really covered when I merge.

## 3. One row per applicant

I grouped by `SK_ID_CURR` and created 5 features:

| Feature | What it is |
|---|---|
| `bureau_count` | number of past credits |
| `bureau_active` | how many of them are still open |
| `bureau_credit_sum` | total credit amount |
| `bureau_debt_sum` | total debt still owed |
| `bureau_max_overdue` | longest overdue period (days) |

In [7]:
# flag for credits that are still open
bureau["ACTIVE"] = (bureau["CREDIT_ACTIVE"] == "Active").astype(int)

bureau_agg = bureau.groupby("SK_ID_CURR").agg(
    bureau_count=("SK_ID_BUREAU", "count"),
    bureau_active=("ACTIVE", "sum"),
    bureau_credit_sum=("AMT_CREDIT_SUM", "sum"),
    bureau_debt_sum=("AMT_CREDIT_SUM_DEBT", "sum"),
    bureau_max_overdue=("CREDIT_DAY_OVERDUE", "max")
).reset_index()

print(bureau_agg.shape)
bureau_agg.head()

(305811, 6)


,SK_ID_CURR,bureau_count,bureau_active,bureau_credit_sum,bureau_debt_sum,bureau_max_overdue
0,100001,7,3,1453365.000,596686.5,0
1,100002,8,2,865055.565,245781.0,0
2,100003,4,1,1017400.500,0.0,0
3,100004,2,0,189037.800,0.0,0
4,100005,3,2,657126.000,568408.5,0


Now I have **305,811 rows (one per applicant) and 6 columns** (the ID plus the 5 features), so it can be joined with the main table safely.

One thing to be careful about: `sum()` in pandas treats missing values as 0. A lot of `AMT_CREDIT_SUM_DEBT` values were missing in the raw data, so a `bureau_debt_sum` of 0 can mean "no debt" or "unknown". I can't tell the two apart with this feature, so to check later.

## 4. Does it relate to default?

I merged the new features with `TARGET` and looked at the correlation.

In [9]:
# does it relate to default? (need TARGET from the main table)
app = pd.read_csv(path + "application_train.csv")
test = app[["SK_ID_CURR", "TARGET"]].merge(bureau_agg, on="SK_ID_CURR", how="left")
print(test.corr()["TARGET"])

print(test.drop(columns="SK_ID_CURR").corr()["TARGET"])

SK_ID_CURR           -0.002108
TARGET                1.000000
bureau_count          0.004056
bureau_active         0.067128
bureau_credit_sum    -0.014057
bureau_debt_sum       0.007144
bureau_max_overdue    0.005493
Name: TARGET, dtype: float64
TARGET                1.000000
bureau_count          0.004056
bureau_active         0.067128
bureau_credit_sum    -0.014057
bureau_debt_sum       0.007144
bureau_max_overdue    0.005493
Name: TARGET, dtype: float64


| Feature | Correlation with TARGET |
|---|---|
| bureau_active | 0.067 |
| bureau_credit_sum | -0.014 |
| bureau_debt_sum | 0.007 |
| bureau_max_overdue | 0.005 |
| bureau_count | 0.004 |

- `bureau_active` is the strongest one, but it's still weak. People with more credits that are *still open* default slightly more.
- The total number of credits (`bureau_count`) says almost nothing by itself.
- All of them are under 0.07, so on their own these features are weak. They might help more together with other features, or as ratios (for example debt divided by total credit).
- **44,020 applicants (about 14%) have no bureau history.** They get `NaN` after the merge. I'll fill the counts with 0 for these people in the merge notebook.
- `SK_ID_CURR` shows up in the correlation list, but it's just an ID, so that number means nothing and I should ignore it.

## Summary

**What I found**
- 1.7 million past credits, grouped into one row per applicant
- About 14% of the train applicants have no bureau history
- The correlations with default are all weak (max 0.067, `bureau_active`)

**To check later**
- Whether 0 in `bureau_debt_sum` means "no debt" or "missing"
- Ratio features like debt / credit sum

**Next:** same thing for `previous_application.csv`